<a href="https://colab.research.google.com/github/Long-aa/AI_Canh_Bao_Nga_Nguoi_Gia/blob/origin%2FLong/Train_AI_Ng%C3%A3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==========================================
# KHỐI CODE TỰ ĐỘNG CÀI ĐẶT SAFEGUARD AI
# ==========================================
import os
import socket
import subprocess

print("1. Đang giải nén mã nguồn...")
!unzip -q -o backend_server.zip

print("2. Đang cài đặt thư viện...")
!pip install -q fastapi uvicorn mediapipe opencv-python-headless sqlalchemy python-multipart python-dotenv passlib[bcrypt] python-jose[cryptography] pyngrok supabase
!npm install -g localtunnel -s > /dev/null 2>&1

print("3. Tự động vá các lỗi hệ thống...")
db_file = '/content/backend_server/app/models/database.py'
if os.path.exists(db_file):
    with open(db_file, 'r') as f: lines = f.readlines()
    with open(db_file, 'w') as f:
        for line in lines:
            if 'SQLALCHEMY_DATABASE_URL =' in line or 'SQLALCHEMY_DATABASE_URL=' in line:
                f.write("SQLALCHEMY_DATABASE_URL = 'sqlite:///./test.db'\n")
            else: f.write(line)

pose_file = '/content/backend_server/app/ai/pose_extractor.py'
if os.path.exists(pose_file):
    with open(pose_file, 'r') as f: content = f.read()
    content = content.replace("model_asset_path='app/ai/pose_landmarker.task'", "model_asset_path='/content/backend_server/app/ai/pose_landmarker.task'")
    with open(pose_file, 'w') as f: f.write(content)

print("4. Khởi tạo dữ liệu...")
%cd /content/backend_server
!python seed_db.py

print("\n" + "="*50)
colab_ip = socket.gethostbyname(socket.gethostname())
print("🚀 ĐANG KHỞI ĐỘNG SERVER AI...")
print(f"🔑 MẬT KHẨU XÁC NHẬN CỦA BẠN: {colab_ip}")
print("="*50 + "\n")

# Chạy Server AI ngầm
get_ipython().system_raw('python server.py &')

# Chạy Tunnel hiển thị trực tiếp (Sẽ in ra link ngay)
!lt --port 8001

1. Đang giải nén mã nguồn...
2. Đang cài đặt thư viện...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 68.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.8/150.8 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 525.6/525.6 kB 24.7 MB/s eta 0:00:00
3. Tự động vá các lỗi hệ thống...
4. Khởi tạo dữ liệu...
/content/backend_server
/content/backend_server/seed_db.py:17: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  Device(device_id="EDG-001", name="Cam Phòng Khách", location="Phòng khách - Khu A", model="NVIDIA Jetson Nano", status="online", cpu_usage=

In [2]:
import os
from dotenv import load_dotenv
import requests

# Tải biến môi trường
load_dotenv('/content/backend_server/.env')

SUPA_URL = os.getenv("SUPABASE_URL")
SUPA_KEY = os.getenv("SUPABASE_KEY")
BUCKET = os.getenv("SUPABASE_BUCKET", "alerts")

print("🔍 Đang kiểm tra kết nối Supabase...")
print(f"🔗 URL: {SUPA_URL}")
print(f"🔑 Định dạng Key hợp lệ (JWT): {SUPA_KEY.startswith('eyJ')}")

# Gọi thử API kiểm tra Bucket
headers = {"Authorization": f"Bearer {SUPA_KEY}"}
res = requests.get(f"{SUPA_URL}/storage/v1/bucket/{BUCKET}", headers=headers)

if res.status_code == 200:
    print(f"✅ KẾT NỐI THÀNH CÔNG! Đã tìm thấy Bucket '{BUCKET}'")
else:
    print(f"❌ LỖI KẾT NỐI ({res.status_code}): {res.text}")


🔍 Đang kiểm tra kết nối Supabase...
🔗 URL: https://yyjeugsyleqwidecmgyz.supabase.co
🔑 Định dạng Key hợp lệ (JWT): True
✅ KẾT NỐI THÀNH CÔNG! Đã tìm thấy Bucket 'alerts'


In [3]:
import os

file_path = '/content/backend_server/app/ai/pose_extractor.py'
with open(file_path, 'r') as f:
    content = f.read()

# Mã lệnh thay thế xương cơ bản thành Full 33 điểm xương chi tiết
old_connections = """        connections = [
            (11, 12), (11, 23), (12, 24), (23, 24), # Torso
            (11, 13), (13, 15), # Left arm
            (12, 14), (14, 16), # Right arm
            (23, 25), (25, 27), # Left leg
            (24, 26), (26, 28)  # Right leg
        ]"""

new_connections = """        connections = [
            # Khuôn mặt (Mắt, mũi, tai, miệng)
            (0, 1), (1, 2), (2, 3), (3, 7),
            (0, 4), (4, 5), (5, 6), (6, 8),
            (9, 10),
            # Thân mình
            (11, 12), (11, 23), (12, 24), (23, 24),
            # Tay trái (bao gồm các đốt ngón tay)
            (11, 13), (13, 15), (15, 17), (15, 19), (15, 21), (17, 19),
            # Tay phải (bao gồm các đốt ngón tay)
            (12, 14), (14, 16), (16, 18), (16, 20), (16, 22), (18, 20),
            # Chân trái (bao gồm mũi chân và gót chân)
            (23, 25), (25, 27), (27, 29), (27, 31), (29, 31),
            # Chân phải (bao gồm mũi chân và gót chân)
            (24, 26), (26, 28), (28, 30), (28, 32), (30, 32)
        ]"""

new_content = content.replace(old_connections, new_connections)

with open(file_path, 'w') as f:
    f.write(new_content)

print("✨ ĐÃ NÂNG CẤP THÀNH CÔNG LÊN MÔ HÌNH 33 ĐIỂM CHI TIẾT!")


✨ ĐÃ NÂNG CẤP THÀNH CÔNG LÊN MÔ HÌNH 33 ĐIỂM CHI TIẾT!


In [4]:
import os

print("1. Nâng cấp luồng dữ liệu Server...")
server_file = '/content/backend_server/server.py'
with open(server_file, 'r') as f: server_content = f.read()

old_encode = """                    _, buffer = cv2.imencode('.jpg', frame, [cv2.IMWRITE_JPEG_QUALITY, 40])
                    annotated_data = f"data:image/jpeg;base64,{base64.b64encode(buffer).decode()}"

                    if device_id in active_streams:
                        for client in list(active_streams[device_id]):
                            try: await client.send_text(annotated_data)
                            except: active_streams[device_id].remove(client)"""

new_encode = """                    _, buffer = cv2.imencode('.jpg', frame, [cv2.IMWRITE_JPEG_QUALITY, 40])
                    annotated_data = json.dumps({"type": "frame", "data": f"data:image/jpeg;base64,{base64.b64encode(buffer).decode()}"})

                    if device_id in active_streams:
                        for client in list(active_streams[device_id]):
                            try: await client.send_text(annotated_data)
                            except: active_streams[device_id].remove(client)"""

old_alert = """                                        await manager.broadcast({"type": "fall_alert", "data": {"id": new_alert.id, "confidence": confidence}})
                                except: pass"""

new_alert = """                                        await manager.broadcast({"type": "fall_alert", "data": {"id": new_alert.id, "confidence": confidence}})
                                        # Bắn thông báo cảnh báo về giao diện AI History
                                        alert_msg = json.dumps({
                                            "type": "alert",
                                            "level": "warning",
                                            "message": f"🚨 CẢNH BÁO: Phát hiện người Ngã / Đột quỵ! ({int(confidence*100)}%)"
                                        })
                                        for client in list(active_streams[device_id]):
                                            try: await client.send_text(alert_msg)
                                            except: pass
                                except: pass"""

server_content = server_content.replace(old_encode, new_encode).replace(old_alert, new_alert)
with open(server_file, 'w') as f: f.write(server_content)

print("2. Nâng cấp bộ não AI phát hiện đột quỵ...")
action_file = '/content/backend_server/app/ai/action_recognizer.py'
with open(action_file, 'r') as f: action_content = f.read()

old_heuristic = """        if self.last_y_pos is not None:
            velocity = current_center_y - self.last_y_pos # Positive means moving down

            # CONDITION A: Rapid downward movement followed by horizontal state
            # full_ratio < 1.0 means width > height (lying down)
            if full_ratio < 0.8:
                # If moving down or already low
                if velocity > 0.02 or current_center_y > 0.5:
                    if head_below_hips or full_ratio < 0.6:
                        self.fall_counter += 1

                if self.fall_counter > 3: # Need to stay in this state for a bit
                    is_falling = True
                    confidence = min(0.9, 0.5 + (1.0 - full_ratio))
            else:
                # Gradually decrease counter if person is upright
                self.fall_counter = max(0, self.fall_counter - 1)"""

new_heuristic = """        if self.last_y_pos is not None:
            velocity = current_center_y - self.last_y_pos
            head_y = y[0]
            ankle_y = (y[27] + y[28]) / 2 if (y[27] > 0 and y[28] > 0) else max_y

            # ĐIỀU KIỆN: Ngã sõng soài HOẶC Đầu hạ thấp quá nhanh HOẶC Đầu nằm sát gót chân
            if full_ratio < 0.8 or velocity > 0.04 or head_y > 0.7 or abs(ankle_y - head_y) < 0.25:
                if velocity > 0.02 or head_y > 0.6 or full_ratio < 0.8:
                    self.fall_counter += 1
            else:
                self.fall_counter = max(0, self.fall_counter - 1)

            if self.fall_counter >= 3:
                is_falling = True
                confidence = min(0.95, 0.6 + (self.fall_counter * 0.1))"""

action_content = action_content.replace(old_heuristic, new_heuristic)
with open(action_file, 'w') as f: f.write(action_content)

print("✅ ĐÃ NÂNG CẤP XONG!")


1. Nâng cấp luồng dữ liệu Server...
2. Nâng cấp bộ não AI phát hiện đột quỵ...
✅ ĐÃ NÂNG CẤP XONG!


In [ ]:
import os
import numpy as np

print("Nâng cấp Chống Báo Động Giả cho AI (Phiên bản Đa người - NÂNG CẤP MẠNH)...")
action_file = '/content/backend_server/app/ai/action_recognizer.py'

if not os.path.exists(action_file):
    print(f"❌ Lỗi: Không tìm thấy file tại {action_file}")
else:
    with open(action_file, 'r') as f:
        action_content = f.read()

    # Tìm vị trí hàm heuristic phiên bản đa người
    start_pattern = "    def _predict_heuristic(self, person_id):"
    end_pattern = "    def reset_sequence(self):"

    start_idx = action_content.find(start_pattern)
    end_idx = action_content.find(end_pattern)

    new_method = """    def _predict_heuristic(self, person_id):
        state = self.person_states[person_id]
        if not state["pose_sequence"]:
            return "normal", 0.0
            
        current_pose = np.asarray(state["pose_sequence"][-1])
        landmarks = current_pose.reshape(33, 3)
        x = landmarks[:, 0]
        y = landmarks[:, 1]
        
        head_x, head_y = x[0], y[0]
        shoulder_x = (x[11] + x[12]) / 2
        shoulder_y = (y[11] + y[12]) / 2
        hip_x = (x[23] + x[24]) / 2
        hip_y = (y[23] + y[24]) / 2
        ankle_x = (x[27] + x[28]) / 2
        ankle_y = (y[27] + y[28]) / 2
        
        # Lấy tọa độ 2D của các khớp để tính góc
        l_shoulder = landmarks[11][:2]
        r_shoulder = landmarks[12][:2]
        l_hip = landmarks[23][:2]
        r_hip = landmarks[24][:2]
        l_knee = landmarks[25][:2]
        r_knee = landmarks[26][:2]
        l_ankle = landmarks[27][:2]
        r_ankle = landmarks[28][:2]

        def calculate_angle(p1, p2, p3):
            v1 = p1 - p2
            v2 = p3 - p2
            cos_theta = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-6)
            angle = np.arccos(np.clip(cos_theta, -1.0, 1.0))
            return np.degrees(angle)

        left_knee_angle = calculate_angle(l_hip, l_knee, l_ankle)
        right_knee_angle = calculate_angle(r_hip, r_knee, r_ankle)
        left_hip_angle = calculate_angle(l_shoulder, l_hip, l_knee)
        right_hip_angle = calculate_angle(r_shoulder, r_hip, r_knee)

        knee_angle = (left_knee_angle + right_knee_angle) / 2
        hip_angle = (left_hip_angle + right_hip_angle) / 2

        lower_body_visible = abs(ankle_y - hip_y) > 0.1 and hip_y < 0.95
        
        min_y, max_y = np.min(y), np.max(y)
        min_x, max_x = np.min(x), np.max(x)
        full_height = max_y - min_y
        full_width = max_x - min_x
        full_ratio = full_height / (full_width + 1e-6)
        
        # Góc thân người so với phương đứng (Torso Vertical Angle)
        torso_vector = np.array([shoulder_x - hip_x, shoulder_y - hip_y])
        torso_angle_vertical = np.degrees(np.arctan2(abs(torso_vector[0]), abs(torso_vector[1])))
        
        current_center_y = (shoulder_y + hip_y) / 2
        
        # Tính toán vận tốc rơi trọng tâm và rơi của đầu
        cg_velocities = []
        head_velocities = []
        torso_angles = []
        upright_states = []
        
        for i in range(1, len(state["pose_sequence"])):
            prev_p = np.asarray(state["pose_sequence"][i-1]).reshape(33, 3)
            curr_p = np.asarray(state["pose_sequence"][i]).reshape(33, 3)
            
            prev_cg_y = (prev_p[11, 1] + prev_p[12, 1] + prev_p[23, 1] + prev_p[24, 1]) / 4
            curr_cg_y = (curr_p[11, 1] + curr_p[12, 1] + curr_p[23, 1] + curr_p[24, 1]) / 4
            cg_velocities.append(curr_cg_y - prev_cg_y)
            
            head_velocities.append(curr_p[0, 1] - prev_p[0, 1])
            
            # Tính góc thân để phát hiện thay đổi hướng nhanh
            prev_shoulder_x = (prev_p[11, 0] + prev_p[12, 0]) / 2
            prev_shoulder_y = (prev_p[11, 1] + prev_p[12, 1]) / 2
            prev_hip_x = (prev_p[23, 0] + prev_p[24, 0]) / 2
            prev_hip_y = (prev_p[23, 1] + prev_p[24, 1]) / 2
            prev_torso_vector = np.array([prev_shoulder_x - prev_hip_x, prev_shoulder_y - prev_hip_y])
            prev_torso_angle = np.degrees(np.arctan2(abs(prev_torso_vector[0]), abs(prev_torso_vector[1])))
            torso_angles.append(abs(torso_angle_vertical - prev_torso_angle))
            
            # Phát hiện trạng thái đứng thẳng trong quá khứ
            prev_ratio = (np.max(prev_p[:, 1]) - np.min(prev_p[:, 1])) / (np.max(prev_p[:, 0]) - np.min(prev_p[:, 0]) + 1e-6)
            prev_torso_angle = np.degrees(np.arctan2(abs(prev_shoulder_x - prev_hip_x), abs(prev_shoulder_y - prev_hip_y)))
            upright_states.append(prev_ratio > 0.75 and prev_torso_angle < 35)
            
        max_drop_velocity = max(cg_velocities) if cg_velocities else 0.0
        max_head_drop_velocity = max(head_velocities) if head_velocities else 0.0
        avg_drop_velocity = np.mean(cg_velocities) if cg_velocities else 0.0
        
        # Tính gia tốc (jerk)
        accelerations = []
        for i in range(1, len(cg_velocities)):
            accelerations.append(abs(cg_velocities[i] - cg_velocities[i-1]))
        max_acceleration = max(accelerations) if accelerations else 0.0
        
        # Tính tốc độ thay đổi góc thân
        max_torso_angle_change = max(torso_angles) if torso_angles else 0.0
        
        # Phát hiện chuyển đổi nhanh từ đứng sang nằm (trong 5-10 khung hình gần nhất)
        recent_frames = min(10, len(upright_states))
        was_upright_recently = any(upright_states[-recent_frames:]) if upright_states else False
        rapid_transition = was_upright_recently and is_lying and max_torso_angle_change > 20
        
        # Phát hiện va đập đầu (đầu dừng đột ngột sau khi rơi nhanh)
        head_impact = False
        if len(head_velocities) >= 3:
            for i in range(len(head_velocities) - 2):
                # Tìm pattern: rơi nhanh rồi dừng đột ngột
                if head_velocities[i] > 0.015 and abs(head_velocities[i+1]) < 0.005:
                    head_impact = True
                    break
        
        # Phát hiện tư thế tay chân bất thường khi ngã
        l_elbow_y = y[13]
        r_elbow_y = y[14]
        l_wrist_y = y[15]
        r_wrist_y = y[16]
        arms_spread = abs(l_wrist_y - r_wrist_y) > 0.15 or abs(l_elbow_y - r_elbow_y) > 0.15
        
        # Ngồi ghế
        is_sitting = (knee_angle < 135 and knee_angle > 70) and (hip_angle < 135 and hip_angle > 70) and (torso_angle_vertical < 35) and (full_ratio > 0.75)
        
        # Trạng thái nằm ngang
        is_lying = (full_ratio < 0.65) or (torso_angle_vertical > 50)
        
        # Xác định vùng cao hơn sàn nhà (Giường, Sofa, Võng)
        is_elevated = (current_center_y < 0.68) or (hip_y < 0.72 and shoulder_y < 0.72)
        
        # Nhận diện tư thế nằm võng
        is_hammock_posture = is_lying and (hip_y > head_y + 0.04) and (hip_y > ankle_y + 0.04)
        
        # Vấp ngã bậc thang
        is_tripping_stairs = (
            (max_drop_velocity > 0.04 or max_head_drop_velocity > 0.04) and 
            (head_y >= (hip_y - 0.1)) and 
            (torso_angle_vertical > 45) and
            not is_elevated
        )
        
        # Té ngã bình thường - NÂNG CẤP MẠNH:
        is_falling_general = (
            is_lying and
            not is_sitting and
            not is_hammock_posture and
            (
                (max_drop_velocity > 0.012 or max_head_drop_velocity > 0.012) or
                (max_acceleration > 0.010) or
                (max_torso_angle_change > 12) or
                rapid_transition or
                head_impact or
                (arms_spread and max_drop_velocity > 0.008)
            )
        )
        
        # Đi ngủ - SIẾT CHẶT HƠN:
        is_sleeping = is_lying and (
            (is_elevated and max_drop_velocity < 0.010 and max_acceleration < 0.005 and max_torso_angle_change < 5 and not rapid_transition and not head_impact) or
            (is_hammock_posture and max_drop_velocity < 0.010 and max_acceleration < 0.005 and not rapid_transition) or
            (max_drop_velocity < 0.008 and max_head_drop_velocity < 0.008 and max_acceleration < 0.004 and max_torso_angle_change < 5 and not rapid_transition and not head_impact)
        )

        is_falling = False
        is_tripped = False
        confidence = 0.0
        
        if "has_fallen" not in state:
            state["has_fallen"] = False
        if "fall_type" not in state:
            state["fall_type"] = None
        
        if state["last_y_pos"] is not None:
            if is_tripping_stairs:
                state["fall_counter"] += 2
            elif is_falling_general:
                # Tăng counter nhanh hơn nếu có dấu hiệu mạnh
                if rapid_transition or head_impact or max_drop_velocity > 0.025:
                    state["fall_counter"] += 2
                else:
                    state["fall_counter"] += 1
            else:
                if state["has_fallen"] and is_lying:
                    pass
                else:
                    state["fall_counter"] = max(0, state["fall_counter"] - 1)
                
            if state["fall_counter"] >= 2:  # Giảm từ 3 xuống 2 để nhạy hơn
                if is_tripping_stairs or (state["fall_counter"] >= 3 and torso_angle_vertical > 60) or (state.get("fall_type") == "fall_stairs" and is_lying):
                    is_tripped = True
                    state["fall_type"] = "fall_stairs"
                else:
                    is_falling = True
                    state["fall_type"] = "fall"
                state["has_fallen"] = True
                confidence = min(0.98, 0.70 + (state["fall_counter"] * 0.08))
            else:
                state["has_fallen"] = False
                state["fall_type"] = None
        
        state["last_y_pos"] = current_center_y
        
        if is_tripped:
            return "fall_stairs", confidence
        elif is_falling:
            return "fall", confidence
        elif is_sitting:
            return "sitting", 0.85
        elif is_sleeping:
            return "sleeping", 0.90
            
        return "normal", 0.0
"""

    if start_idx != -1 and end_idx != -1:
        action_content = action_content[:start_idx] + new_method + action_content[end_idx:]
        with open(action_file, 'w') as f: f.write(action_content)
        print("✅ ĐÃ NÂNG CẤP THÀNH CÔNG! AI ĐA NGƯỜI VỚI PHÁT HIỆN NGÃ MẠNH HƠN.")
    else:
        print("❌ Lỗi: Không tìm thấy cấu trúc hàm trong file. Vui lòng kiểm tra lại file action_recognizer.py")

Nâng cấp Chống Báo Động Giả cho AI (Phiên bản Đa người)...
✅ ĐÃ NÂNG CẤP THÀNH CÔNG! AI ĐA NGƯỜI SẼ HOẠT ĐỘNG CHÍNH XÁC HƠN.


In [6]:
import os
import gc

print("1. Cài đặt các thư viện lõi (face_recognition)...")
!pip install -q face_recognition

print("2. Đang cấu trúc lại Bộ não AI...")

# --- PHẦN 1: SỬA LỖI ACTION RECOGNIZER (CHỐNG CRASH) ---
action_file = '/content/backend_server/app/ai/action_recognizer.py'
if os.path.exists(action_file):
    with open(action_file, 'r') as f: action_content = f.read()
    # Thêm kiểm tra size để tránh lỗi reshape khi không có người
    if 'if current_pose.size == 0' not in action_content:
        action_content = action_content.replace(
            'current_pose = np.asarray(state["pose_sequence"][-1])',
            'current_pose = np.asarray(state["pose_sequence"][-1])\n        if current_pose.size == 0: return "normal", 0.0'
        )
    with open(action_file, 'w') as f: f.write(action_content)
    print("✅ Đã nâng cấp ActionRecognizer (Chống crash mảng rỗng)")

# --- PHẦN 2: TẠO MODULE NHẬN DIỆN KHUÔN MẶT ---
face_code = """
import face_recognition
import cv2
import numpy as np
import os
import threading
from app.models.database import get_db, ElderlyProfile

class FaceRecognizerAI:
    def __init__(self):
        self.known_face_encodings = []
        self.known_face_names = []
        self.load_profiles()
        self.process_every_n_frames = 15
        self.frame_count = 0
        self.current_face_locations = []
        self.current_face_names = []
        self._is_processing = False

    def load_profiles(self):
        print("FaceRecognizerAI: Loading profiles from database...")
        try:
            import gc
            db = next(get_db())
            profiles = db.query(ElderlyProfile).filter(ElderlyProfile.video_path != None).all()

            for p in profiles:
                video_path = p.video_path
                if video_path.startswith('http'):
                    full_path = video_path
                else:
                    if video_path.startswith('/'):
                        video_path = video_path[1:] # Remove leading slash for local path
                    full_path = os.path.abspath(os.path.join(os.getcwd(), video_path))
                    if not os.path.exists(full_path):
                        print(f"FaceRecognizerAI: Video not found for {p.name} at {full_path}")
                        continue

                print(f"FaceRecognizerAI: Processing video for {p.name} at {full_path}...")
                cap = cv2.VideoCapture(full_path)
                success, frame = cap.read()
                frames_checked = 0
                face_found = False

                # Scan up to 20 frames, check every 3rd frame
                while success and frames_checked < 20:
                    if frames_checked % 3 == 0:
                        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                        # Use small frame for encoding speed
                        small_frame = cv2.resize(rgb_frame, (0, 0), fx=0.5, fy=0.5)
                        encodings = face_recognition.face_encodings(small_frame)

                        if encodings:
                            self.known_face_encodings.append(encodings[0])
                            self.known_face_names.append(p.name)
                            print(f"FaceRecognizerAI: Successfully learned face for {p.name}")
                            face_found = True
                            break
                    success, frame = cap.read()
                    frames_checked += 1
                cap.release()
                if not face_found:
                    print(f"FaceRecognizerAI: No face found in video for {p.name}")

            db.close()
            gc.collect() # Free up RAM
        except Exception as e:
            print(f"FaceRecognizerAI: Error loading profiles: {e}")

    def _bg_process_frame(self, frame):
        try:
            # Resize frame for faster processing (0.25 size reduces pixel area by 16x)
            small_frame = cv2.resize(frame, (0, 0), fx=0.25, fy=0.25)
            rgb_small_frame = cv2.cvtColor(small_frame, cv2.COLOR_BGR2RGB)

            locations = face_recognition.face_locations(rgb_small_frame)
            face_encodings = face_recognition.face_encodings(rgb_small_frame, locations)

            names = []
            for face_encoding in face_encodings:
                matches = face_recognition.compare_faces(self.known_face_encodings, face_encoding, tolerance=0.5)
                name = "Unknown"

                if True in matches:
                    first_match_index = matches.index(True)
                    name = self.known_face_names[first_match_index]

                names.append(name)

            # Update atomically
            self.current_face_locations = locations
            self.current_face_names = names
        except Exception as e:
            print(f"FaceRecognizerAI background processing error: {e}")
        finally:
            self._is_processing = False

    def process_frame(self, frame):
        self.frame_count += 1

        # Only process every N frames and only if not already processing
        if not self._is_processing and (self.frame_count == 1 or self.frame_count % self.process_every_n_frames == 0):
            self._is_processing = True
            frame_clone = frame.copy()
            threading.Thread(target=self._bg_process_frame, args=(frame_clone,), daemon=True).start()

        # Draw results on the frame (uses last computed locations and names)
        for (top, right, bottom, left), name in zip(self.current_face_locations, self.current_face_names):
            # Scale back up (x4 since we processed at 0.25 scale)
            top *= 4; right *= 4; bottom *= 4; left *= 4

            # Draw box
            cv2.rectangle(frame, (left, top), (right, bottom), (0, 255, 255), 2)

            # Draw label
            cv2.rectangle(frame, (left, top - 35), (right, top), (0, 255, 255), cv2.FILLED)
            font = cv2.FONT_HERSHEY_DUPLEX
            cv2.putText(frame, name, (left + 6, top - 6), font, 0.6, (0, 0, 0), 1)

        return frame

"""
with open('/content/backend_server/app/ai/face_recognizer.py', 'w') as f: f.write(face_code)
print("✅ Đã tạo Module FaceRecognizerAI")

# --- PHẦN 3: VÁ LỖI SERVER (DỮ LIỆU & IMPORT) ---
server_file = '/content/backend_server/server.py'
if os.path.exists(server_file):
    with open(server_file, 'r') as f: server_content = f.read()

    # 1. Thêm Import nếu chưa có
    if "from app.ai.face_recognizer" not in server_content:
        server_content = server_content.replace(
            "from app.ai.pose_extractor import PoseExtractor",
            "from app.ai.pose_extractor import PoseExtractor\nfrom app.ai.face_recognizer import FaceRecognizerAI"
        )

    # 2. Sửa lỗi định dạng gửi ảnh (Quan trọng: Sửa 'image' thành 'data' và thêm tiền tố base64)
    if '"image": b64_img' in server_content:
        server_content = server_content.replace(
            'json.dumps({"type": "frame", "image": b64_img})',
            'json.dumps({"type": "frame", "data": f"data:image/jpeg;base64,{b64_img}"})'
        )

    # 3. Chèn AI Face vào luồng xử lý chính
    if '"face": FaceRecognizerAI()' not in server_content:
        server_content = server_content.replace(
            '"pose": PoseExtractor(),',
            '"pose": PoseExtractor(),\n            "face": FaceRecognizerAI(),'
        )

    if 'processor["face"].process_frame(frame)' not in server_content:
        server_content = server_content.replace(
            'frame = processor["pose"].draw_pose(frame, pose_landmarks_obj)',
            'frame = processor["pose"].draw_pose(frame, pose_landmarks_obj)\n                    frame = processor["face"].process_frame(frame)'
        )

    with open(server_file, 'w') as f: f.write(server_content)
    print("✅ Đã vá lỗi Server (Định dạng ảnh & Kết nối AI Face)")

print("\n" + "="*50)
print("🚀 TỔNG HỢP HOÀN TẤT! HỆ THỐNG ĐÃ SẴN SÀNG NHẬN DIỆN.")
print("="*50)


1. Cài đặt các thư viện lõi (face_recognition)...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.1/100.1 MB 8.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
2. Đang cấu trúc lại Bộ não AI...
✅ Đã nâng cấp ActionRecognizer (Chống crash mảng rỗng)
✅ Đã tạo Module FaceRecognizerAI
✅ Đã vá lỗi Server (Định dạng ảnh & Kết nối AI Face)

🚀 TỔNG HỢP HOÀN TẤT! HỆ THỐNG ĐÃ SẴN SÀNG NHẬN DIỆN.


In [7]:
# Đã tích hợp đầy đủ tính năng (Zalo webhook, Face, Pose, Stairs, Video uploads) vào server.py trong zip
print('✅ Sử dụng file server.py chính thức từ backend_server.zip (không ghi đè)')

✅ Sử dụng file server.py chính thức từ backend_server.zip (không ghi đè)


In [ ]:
# ==========================================
# KHỐI CODE COLAB SỬ DỤNG NGROK (CHỐNG LỖI WEBSOCKET)
# ==========================================
import os
import threading

print("0. Dọn dẹp các đường hầm Ngrok bị kẹt...")
!pkill ngrok

print("1. Cài đặt Ngrok...")
!pip install -q pyngrok

# Vá lỗi đường dẫn AI
pose_file = '/content/backend_server/app/ai/pose_extractor.py'
if os.path.exists(pose_file):
    with open(pose_file, 'r') as f: content = f.read()
    content = content.replace("model_asset_path='app/ai/pose_landmarker.task'", "model_asset_path='/content/backend_server/app/ai/pose_landmarker.task'")
    with open(pose_file, 'w') as f: f.write(content)

print("\n2. Khởi động Ngrok...")
from pyngrok import ngrok

# Tắt ngrok an toàn từ thư viện
ngrok.kill()

# ===== THAY TOKEN CỦA BẠN VÀO ĐÂY =====
ngrok.set_auth_token("3Dx69CxZUaePR4N84O4CmqyYhcf_oBk9oxCWPS39T6tUd5bc")
# ======================================

public_url = ngrok.connect(8001, domain="nemesis-modify-mollusk.ngrok-free.dev").public_url
print("\n" + "="*50)
print(f"🚀 LINK MỚI CỦA BẠN LÀ: {public_url}")
print("="*50 + "\n")

print("3. Khởi động Server AI...")
%cd /content/backend_server
!python server.py


0. Dọn dẹp các đường hầm Ngrok bị kẹt...
1. Cài đặt Ngrok...

2. Khởi động Ngrok...

🚀 LINK MỚI CỦA BẠN LÀ: https://nemesis-modify-mollusk.ngrok-free.dev

3. Khởi động Server AI...
/content/backend_server
2026-05-20 05:50:05.086483: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-05-20 05:50:05.301606: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Connected to Supabase project: https://yyjeugsyleqwidecmgyz.supabase.co
INFO:     Started server process [12103]
INFO:     Waiting for application startup.
Database tables created/verified.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8001 (Press CTRL+C to quit)
INFO:     42.113.45.126:0 - "OPTI